# XGBoost Localization: History Depth Sweep Notebook

This notebook performs a **sweep over different history depths ($h$)** using **XGBoost** for CSI-based UE localization on the 25x25 grid (4 m spacing).

**Goal:**
- Validate the thesis that incorporating sequential history improves localization accuracy.
- Observe the trend of 3D MAE as history depth increases from $h=0$ to $h=10$.

**Feature Representation:**
- For a history depth $h$, we stack the absolute metrics of the current step $t$ and $h$ previous steps: `[rss_t, sinr_t, aoa_az_t, aoa_el_t, rss_t-1, sinr_t-1, ..., rss_t-h, sinr_t-h, aoa_az_t-h, aoa_el_t-h]`.
- These temporal features are concatenated with the static device profile `[n_antennas, gain, height]` to form a flat feature vector of size `(h+1)*4 + 3`.

In [ ]:
import sys, os, time, warnings, json
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.io import loadmat
from sklearn.preprocessing import StandardScaler
from sklearn.multioutput import MultiOutputRegressor
from xgboost import XGBRegressor

SEED = 42
np.random.seed(SEED)

# Adjust if running from a different directory
NOTEBOOK_DIR = Path(os.getcwd())
PROJECT_ROOT = NOTEBOOK_DIR
for _ in range(10):
    if (PROJECT_ROOT / 'results' / 'grid_localization' / 'grid_25x25').exists():
        break
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        break
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / 'experiments' / '09_grid_localization' / 'src' / 'python'))
from utils.read_jsonc import read_jsonc

import datetime
RUN_TIMESTAMP = datetime.datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
DATA_DIR = PROJECT_ROOT / 'results' / 'grid_localization' / 'grid_25x25' / 'sim_data_ne_bs_2026-06-13_14-54-18'
OUT_DIR  = PROJECT_ROOT / 'results' / 'notebook_experiments' / 'xgboost' / f'xgboost_sweep_{RUN_TIMESTAMP}'
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Project root: {PROJECT_ROOT}')
print(f'Data exists:  {DATA_DIR.exists()}')
print(f'Output dir:   {OUT_DIR}')

## Hyperparameters & Settings

In [ ]:
# ── Sweep Configuration ───────────────────────────────────────────────
H_VALUES      = [0, 1, 2, 3, 4, 5, 7, 10]  # Lags to evaluate
SUBSAMPLE     = 0.30                 # Subsample training data for speed (set 1.0 for full data)
TEST_RATIO    = 0.20                 # Chronological split ratio

# ── XGBoost Model Settings (Tuned from Optuna baseline) ─────────────────
N_ESTIMATORS  = 150
MAX_DEPTH     = 8                    # Tuned depth
LEARNING_RATE = 0.276                # Tuned learning rate
SUBSAMPLE_XGB = 0.873                # Tuned row subsampling
COLSAMPLE_BYTREE = 0.907             # Tuned feature subsampling
N_JOBS        = -1                   # Use all available cores

# ── Columns ────────────────────────────────────────────────────────────
SIGNAL_COLS   = ['rss', 'sinr', 'aoa_azimuth', 'aoa_elevation']
STATIC_COLS   = ['n_antennas', 'antenna_gain_db', 'ue_height']
TARGET_COLS   = ['target_x', 'target_y', 'target_z']
GRID_SPACING  = 4.0                  # meters

## Data Loading

In [ ]:
from pipelines.multi_user_pipeline_regression import load_all_users

print('Loading raw data...')
df_raw = load_all_users(DATA_DIR)
print(f'Loaded {len(df_raw):,} steps across all users.')

## Data Split & Preprocessing

In [ ]:
from pipelines.multi_user_pipeline_regression import make_split, _read_bs_position_3d

bs_pos = np.array(_read_bs_position_3d(DATA_DIR))
print(f'BS position: {bs_pos} m')
ue_xyz = np.column_stack([df_raw['x_pos'], df_raw['y_pos'], df_raw['ue_height']])
delta  = ue_xyz - bs_pos
df_raw['target_x'], df_raw['target_y'], df_raw['target_z'] = delta[:,0], delta[:,1], delta[:,2]

# Chronological train/test split per user
df_raw = make_split(df_raw, TEST_RATIO)

# Training subsample
train_idxs, test_idxs = [], df_raw[df_raw['split']=='test'].index.tolist()
for uid in df_raw['user_id'].unique():
    utr = df_raw[(df_raw['user_id']==uid) & (df_raw['split']=='train')].sort_values('step_index')
    train_idxs.extend(utr.index[:int(len(utr)*SUBSAMPLE)].tolist())

df_train = df_raw.loc[train_idxs].copy()
df_test  = df_raw.loc[test_idxs].copy()
print(f'Train samples: {len(df_train):,}  |  Test samples: {len(df_test):,}')

## Feature Sequence Builder

In [ ]:
def build_sequence_features(df, h):
    """Slices chronological history and flattens it for tree regression."""
    all_seq, all_static, all_targets = [], [], []
    for uid in sorted(df['user_id'].unique()):
        udf     = df[df['user_id']==uid].sort_values('step_index').reset_index(drop=True)
        sigs    = udf[SIGNAL_COLS].values.astype(np.float32)
        statics = udf[STATIC_COLS].values.astype(np.float32)
        targets = udf[TARGET_COLS].values.astype(np.float32)
        for i in range(h, len(udf)):
            all_seq.append(sigs[i-h : i+1])  # [h+1, n_signals]
            all_static.append(statics[i])
            all_targets.append(targets[i])
            
    # Convert sequences to flat array
    all_seq = np.stack(all_seq)      # [N, h+1, n_signals]
    all_static = np.stack(all_static)  # [N, n_static]
    all_targets = np.stack(all_targets) # [N, 3]
    
    N = all_seq.shape[0]
    all_seq_flat = all_seq.reshape(N, -1)
    return all_seq_flat, all_static, all_targets

## 7B. Hyperparameter Tuning with Optuna (Optional)

Run this cell to optimize hyperparameters specifically for the target history depth (e.g. the maximum history size `H`). This uses Optuna to run a search sequence and output the best combination of depth, learning rate, and subsampling.

In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    max_depth = trial.suggest_int('max_depth', 2, 9)
    learning_rate = trial.suggest_float('learning_rate', 0.01, 0.20, log=True)
    n_estimators = trial.suggest_int('n_estimators', 50, 250, step=50)
    subsample_xgb = trial.suggest_float('subsample', 0.5, 0.9)
    colsample_bytree = trial.suggest_float('colsample_bytree', 0.4, 0.9)
    min_child_weight = trial.suggest_int('min_child_weight', 1, 20)
    reg_alpha = trial.suggest_float('reg_alpha', 0.0, 10.0)
    reg_lambda = trial.suggest_float('reg_lambda', 1.0, 10.0)
    
    # Build features for current max H
    X_seq_tr_t, X_static_tr_t, y_tr_t = build_sequence_features(df_train, H_VALUES[-1])
    X_seq_te_t, X_static_te_t, y_te_t = build_sequence_features(df_test,  H_VALUES[-1])
    
    # Scale
    sig_scaler = StandardScaler()
    X_seq_tr_norm = sig_scaler.fit_transform(X_seq_tr_t)
    X_seq_te_norm = sig_scaler.transform(X_seq_te_t)
    
    X_seq_tr_norm = np.nan_to_num(X_seq_tr_norm, nan=0.0)
    X_seq_te_norm = np.nan_to_num(X_seq_te_norm, nan=0.0)
    
    static_scaler = StandardScaler()
    X_static_tr_norm = static_scaler.fit_transform(X_static_tr_t)
    X_static_te_norm = static_scaler.transform(X_static_te_t)
    
    X_tr = np.column_stack([X_seq_tr_norm, X_static_tr_norm])
    X_te = np.column_stack([X_seq_te_norm, X_static_te_norm])
    
    model = MultiOutputRegressor(XGBRegressor(
        n_estimators=n_estimators,
        max_depth=max_depth,
        learning_rate=learning_rate,
        subsample=subsample_xgb,
        colsample_bytree=colsample_bytree,
        min_child_weight=min_child_weight,
        reg_alpha=reg_alpha,
        reg_lambda=reg_lambda,
        n_jobs=N_JOBS,
        random_state=42
    ))
    model.fit(X_tr, y_tr_t)
    preds = model.predict(X_te)
    mae = np.mean(np.linalg.norm(preds - y_te_t, axis=1))
    return float(mae)
class TuningCallback:
    def __init__(self, n_trials):
        self.n_trials = n_trials
        self.start_time = time.time()
    def __call__(self, study, trial):
        elapsed = time.time() - self.start_time
        completed = trial.number + 1
        avg_time = elapsed / completed
        eta = avg_time * (self.n_trials - completed)
        print(f'Trial {completed:2d}/{self.n_trials:2d} finished | '
              f'MAE: {trial.value:.3f}m (best: {study.best_value:.3f}m) | '
              f'elapsed={elapsed:.0f}s ETA={eta:.0f}s', flush=True)

print(f'Starting Optuna tuning for H={H_VALUES[-1]}...')
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=20, callbacks=[TuningCallback(20)])  # fast 20 trials search
print("\nBest Parameters found:")
for k, v in study.best_params.items():
    print(f'  {k}: {v}')
print(f'Best MAE on test set: {study.best_value:.3f} m')

## Run the History Sweep

In [ ]:
results = []

for h in H_VALUES:
    print(f'\n{chr(61)*50}')
    print(f'Evaluating History Depth h = {h}')
    print(f'{chr(61)*50}')
    
    # 1. Build sequences
    X_seq_tr, X_static_tr, y_tr = build_sequence_features(df_train, h)
    X_seq_te, X_static_te, y_te = build_sequence_features(df_test,  h)
    
    # 2. Normalize inputs
    sig_scaler = StandardScaler()
    X_seq_tr_norm = sig_scaler.fit_transform(X_seq_tr)
    X_seq_te_norm = sig_scaler.transform(X_seq_te)
    
    X_seq_tr_norm = np.nan_to_num(X_seq_tr_norm, nan=0.0)
    X_seq_te_norm = np.nan_to_num(X_seq_te_norm, nan=0.0)
    
    static_scaler = StandardScaler()
    X_static_tr_norm = static_scaler.fit_transform(X_static_tr)
    X_static_te_norm = static_scaler.transform(X_static_te)
    
    # Combine flat features
    X_tr = np.column_stack([X_seq_tr_norm, X_static_tr_norm])
    X_te = np.column_stack([X_seq_te_norm, X_static_te_norm])
    
    # 3. Initialize & Train Multi-Output XGBoost
    print(f'  Training XGBoost regressor (depth={MAX_DEPTH}, estimators={N_ESTIMATORS})...')
    t_start = time.time()
    
    # MultiOutputRegressor runs independent regressors for X, Y, Z coordinates
    model = MultiOutputRegressor(XGBRegressor(
        n_estimators=N_ESTIMATORS,
        max_depth=MAX_DEPTH,
        learning_rate=LEARNING_RATE,
        subsample=SUBSAMPLE_XGB,
        colsample_bytree=COLSAMPLE_BYTREE,
        n_jobs=N_JOBS,
        random_state=42
    ))
    
    model.fit(X_tr, y_tr)
    elapsed = time.time() - t_start
    
    # 4. Predict & Evaluate (predicting raw Cartesian values directly)
    preds = model.predict(X_te)
    errors = np.linalg.norm(preds - y_te, axis=1)
    mae_3d = float(np.mean(errors))
    mae_xyz = np.mean(np.abs(preds - y_te), axis=0)
    
    print(f'  Done in {elapsed:.1f}s  |  3D MAE = {mae_3d:.3f} m')
    
    results.append({
        'H': h,
        'FeatureDim': X_tr.shape[1],
        '3D_MAE': mae_3d,
        'X_MAE': mae_xyz[0],
        'Y_MAE': mae_xyz[1],
        'Z_MAE': mae_xyz[2],
        'Time': elapsed
    })

## Analysis and Visualization

In [ ]:
df_res = pd.DataFrame(results)
print('\nSweep Results Summary:')
print(df_res.to_string(index=False))

# Plot the curve
plt.figure(figsize=(8, 4.5))
plt.plot(df_res['H'], df_res['3D_MAE'], marker='o', lw=2, color='orangered', label='XGBoost')
plt.xlabel('History Depth (h)')
plt.ylabel('3D Position MAE (m)')
plt.title('XGBoost Localization Error vs. History Depth')
plt.grid(alpha=0.3)
plt.legend()
plt.savefig(OUT_DIR / 'xgb_history_sweep.png', dpi=150, bbox_inches='tight')
plt.show()

## Generate Report

In [ ]:
import datetime
t_stamp = datetime.datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
report_p = OUT_DIR / f'xgb_sweep_report_{t_stamp}.md'

table_lines = [
    '| History Depth (h) | Feature Dim | 3D MAE (m) | X MAE (m) | Y MAE (m) | Z MAE (m) | Training Time (s) |',
    '| :---: | :---: | :---: | :---: | :---: | :---: | :---: |'
]
for r in results:
    table_lines.append(
        f"| {r['H']} | {r['FeatureDim']} | {r['3D_MAE']:.3f} | {r['X_MAE']:.3f} | {r['Y_MAE']:.3f} | {r['Z_MAE']:.3f} | {r['Time']:.1f} |"
    )
table_content = '\n'.join(table_lines)

report_content = f"""# XGBoost Localization History Sweep Report

* **Timestamp:** {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
* **Run ID:** `xgb_sweep_report_{t_stamp}`

## 1. Configuration
* **Dataset:** 100% full dataset (subsampled to {SUBSAMPLE*100:.1f}% for training)
* **Model:** XGBRegressor (depth={MAX_DEPTH}, estimators={N_ESTIMATORS}, lr={LEARNING_RATE}, subsample={SUBSAMPLE_XGB}, colsample_bytree={COLSAMPLE_BYTREE})

## 2. Sweep Results Table

{{table_content}}

## 3. Analysis
This sweep traces how adding temporal sequence metrics impacts decision tree regression. Decision tree models split space along orthogonal boundaries. By stacking lag variables (rss_t-1, etc.), we provide the tree with multi-dimensional indicators that allow it to approximate physical speed and heading vectors.
"""
report_content = report_content.replace('{table_content}', table_content)
report_p.write_text(report_content, encoding='utf-8')
print(f'Report saved to: {report_p}')